# 01b · Manual conversion sample
**Owner:** Aziz · **Output:** `data/manual/manual_sample.csv` and one file per annotator:
`manual_aziz.csv`, `manual_mouhib.csv`, `manual_iskander.csv`

- **60 comments:** 45 Arabic-script from TSAC **train**, 15 Arabizi from TUNIZI V2. Each annotator gets 15 Arabic + 5 Arabizi.
- **Selection rules:** script decided by `detect_script()`; TSAC texts that also appear in the TSAC test file are skipped, and so are duplicated or label-conflicting texts; 3 to 50 words; balanced `pos`/`neg`; `random_state=42`.
- `original_text` is the text after `fix_text()`, i.e. exactly as it will appear in `master_clean.csv`.

In [1]:
# Works both locally (repo root or notebooks/) and in Colab after cloning the repo.
import os, sys
from pathlib import Path
if Path("notebooks").is_dir():
    ROOT = Path(".").resolve()
elif Path("../notebooks").is_dir():
    ROOT = Path("..").resolve()
else:  # Colab: clone first, e.g. !git clone https://github.com/<owner>/tunisian-sentiment.git
    ROOT = Path("/content/tunisian-sentiment")
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("Repo root:", ROOT)

Repo root: /home/claude/work/tunisian-sentiment


In [2]:
import pandas as pd
from src.load_raw import load_tsac
from src.cleaning_utils import fix_text, has_letter
from src.script_detect import detect_script

SEED = 42
N_AR, N_AZ = 45, 15
ANNOTATORS = ["aziz", "mouhib", "iskander"]
OUT_DIR = ROOT / "data/manual"; OUT_DIR.mkdir(parents=True, exist_ok=True)

def n_words(s): return s.str.split().str.len()

## Arabic-script pool (TSAC train)

In [3]:
train, test = load_tsac("train", ROOT), load_tsac("test", ROOT)
train["text"] = train.text.map(fix_text)
test_texts = set(test.text.map(fix_text))

pool = train[(train.text != "") & train.text.map(has_letter)]
pool = pool[~pool.text.isin(test_texts)]                       # never touch anything that is also in test
pool = pool[~pool.text.duplicated(keep=False)]                 # duplicated / conflicting texts out
pool = pool[pool.text.map(detect_script) == "arabic"]
pool = pool[n_words(pool.text).between(3, 50)]
print("Arabic pool:", len(pool)); print(pool.label.value_counts())

n_pos = (N_AR + 1) // 2                                        # 23 pos + 22 neg
ar = pd.concat([pool[pool.label == "pos"].sample(n_pos, random_state=SEED),
                pool[pool.label == "neg"].sample(N_AR - n_pos, random_state=SEED)])
ar = ar.assign(source="tsac", direction="ar_to_arabizi")

Arabic pool: 4348
label
neg    2257
pos    2091
Name: count, dtype: int64


## Arabizi pool (TUNIZI V2, after cleaning)

In [4]:
tz = pd.read_csv(ROOT / "data/interim/tunizi_clean.csv")
tz = tz[tz.text.map(detect_script) == "arabizi"]
tz = tz[n_words(tz.text).between(3, 50)]
print("Arabizi pool:", len(tz))
n_pos = (N_AZ + 1) // 2                                        # 8 pos + 7 neg
az = pd.concat([tz[tz.label == "pos"].sample(n_pos, random_state=SEED),
                tz[tz.label == "neg"].sample(N_AZ - n_pos, random_state=SEED)])
az = az.assign(direction="arabizi_to_ar")

Arabizi pool: 84300


## Deal to annotators and save
Within each script, rows alternate pos/neg before being dealt round-robin, so every file is close to balanced.

In [5]:
def interleave(df):
    p = df[df.label == "pos"].reset_index(drop=True); n = df[df.label == "neg"].reset_index(drop=True)
    out = []
    for i in range(max(len(p), len(n))):
        if i < len(p): out.append(p.iloc[i])
        if i < len(n): out.append(n.iloc[i])
    return pd.DataFrame(out).reset_index(drop=True)

cols = ["id", "source", "label", "direction", "original_text", "manual_text", "notes", "annotator"]
parts = []
for block in (interleave(ar), interleave(az)):
    block = block.rename(columns={"text": "original_text"})
    block["annotator"] = [ANNOTATORS[i % 3] for i in range(len(block))]
    parts.append(block)
sample = pd.concat(parts, ignore_index=True)
sample["manual_text"] = ""; sample["notes"] = ""
sample = sample[cols]

assert sample.id.is_unique and len(sample) == N_AR + N_AZ
sample.to_csv(OUT_DIR / "manual_sample.csv", index=False, encoding="utf-8")
for a in ANNOTATORS:
    sample[sample.annotator == a].drop(columns="annotator").to_csv(OUT_DIR / f"manual_{a}.csv", index=False, encoding="utf-8")

pd.crosstab(sample.annotator, [sample.direction, sample.label], margins=True)

direction ar_to_arabizi     arabizi_to_ar     All
label               neg pos           neg pos    
annotator                                        
aziz                  7   8             2   3  20
iskander              7   8             2   3  20
mouhib                8   7             3   2  20
All                  22  23             7   8  60

In [6]:
# Quick look at what annotators will see
sample.groupby("direction").head(3)[["id", "direction", "label", "original_text"]]

,id,direction,label,original_text
0,tsac_train_06618,ar_to_arabizi,pos,نعم الأخت ألف تحيه ليك
1,tsac_train_08712,ar_to_arabizi,neg,كلكم خرررررررر. ا
2,tsac_train_00115,ar_to_arabizi,pos,وجيهة مرأة ياسر طموحة
45,tunizi_039977,arabizi_to_ar,pos,ana man marocco o nahbk o namot 3alk
46,tunizi_062456,arabizi_to_ar,neg,nidhaal walahy fheemt lougha belghalet sahby e...
47,tunizi_042089,arabizi_to_ar,pos,mashallah khoya samaraa m3allemm grandissimo
